<!-- NB_DOC_INTRO_V1 -->
# Traitement du Signal — FFT, filtres, wavelets, spectrogramme

> 📚 **Doc thematique** : [docs/06_TS_TDS.md](docs/06_TS_TDS.md) (Time Series & Traitement du Signal)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**Traitement du signal** = analyser et transformer des signaux (audio, vibration, capteurs IoT, ECG...). Cle pour : extraction de features (NLP audio, anomaly detection vibration), denoise, compression.

Ce notebook execute : **FFT** (Fourier), **filtres** (passe-bas, passe-haut, passe-bande), **spectrogramme** (STFT), **wavelets** (Haar, Daubechies), **PSD** (Power Spectral Density), denoise par seuillage.

## Plan

1. Generation de signaux jouet (sinus, bruit, multi-freq)
2. FFT (Fast Fourier Transform) + IFFT
3. Filtres frequentiels (low/high/band-pass)
4. Spectrogramme (STFT) — temps × frequence
5. Wavelets — analyse temps-frequence localisee
6. PSD (Welch's method)
7. Denoising par seuillage
8. Application : detection anomalie
9. Pieges + References


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sgn
from scipy.fft import fft, ifft, fftfreq
import warnings
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

## 1. Signaux jouet

In [ ]:
SAMPLE_RATE = 1000   # 1 kHz
DURATION = 2          # secondes
t = np.linspace(0, DURATION, SAMPLE_RATE * DURATION, endpoint=False)

# Signal 1 : sinusoide pure
s1 = np.sin(2 * np.pi * 5 * t)        # 5 Hz

# Signal 2 : somme de sinusoides
s2 = (np.sin(2 * np.pi * 10 * t)        # 10 Hz
      + 0.5 * np.sin(2 * np.pi * 50 * t)  # 50 Hz
      + 0.3 * np.sin(2 * np.pi * 100 * t))  # 100 Hz

# Signal 3 : s2 + bruit blanc
s3 = s2 + np.random.normal(0, 0.5, len(t))

fig, axes = plt.subplots(3, 1, figsize=(12, 7))
axes[0].plot(t[:500], s1[:500]); axes[0].set_title("Signal 1 : 5 Hz pur")
axes[1].plot(t[:500], s2[:500]); axes[1].set_title("Signal 2 : 10 + 50 + 100 Hz")
axes[2].plot(t[:500], s3[:500]); axes[2].set_title("Signal 3 : signal 2 + bruit")
plt.tight_layout(); plt.show()

## 2. FFT — passer du temps au frequentiel

In [ ]:
def plot_fft(signal, fs, title=""):
    n = len(signal)
    fft_vals = fft(signal)
    freqs = fftfreq(n, 1/fs)
    mask = freqs >= 0   # spectrum symetrique, on garde + uniquement
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(freqs[mask], np.abs(fft_vals[mask]) * 2 / n)
    ax.set(xlabel="Frequency (Hz)", ylabel="Amplitude", title=title, xlim=(0, fs/2))
    plt.show()

plot_fft(s2, SAMPLE_RATE, "FFT signal 2 (peaks attendus a 10, 50, 100 Hz)")
plot_fft(s3, SAMPLE_RATE, "FFT signal 3 (bruite mais peaks visibles)")

## 3. Filtres — Butterworth low/high/band-pass

In [ ]:
def butter_filter(data, fs, cutoff, order=5, btype="low"):
    nyquist = 0.5 * fs
    if isinstance(cutoff, (list, tuple)):
        normalized = [c / nyquist for c in cutoff]
    else:
        normalized = cutoff / nyquist
    b, a = sgn.butter(order, normalized, btype=btype)
    return sgn.filtfilt(b, a, data)

# Garder seulement les frequences < 30 Hz (= isole le 10 Hz)
s3_low = butter_filter(s3, SAMPLE_RATE, cutoff=30, btype="low")

# Garder 40-60 Hz (= isole le 50 Hz)
s3_band = butter_filter(s3, SAMPLE_RATE, cutoff=[40, 60], btype="band")

fig, axes = plt.subplots(3, 1, figsize=(12, 7))
axes[0].plot(t[:500], s3[:500]); axes[0].set_title("Original (s3 bruite)")
axes[1].plot(t[:500], s3_low[:500]); axes[1].set_title("Low-pass < 30 Hz (isole 10 Hz)")
axes[2].plot(t[:500], s3_band[:500]); axes[2].set_title("Band-pass 40-60 Hz (isole 50 Hz)")
plt.tight_layout(); plt.show()

## 4. Spectrogramme (STFT) — vue temps × frequence

In [ ]:
# Signal chirp : frequence augmentant lineairement
chirp = sgn.chirp(t, f0=5, f1=200, t1=DURATION, method="linear")

f, t_spec, Sxx = sgn.spectrogram(chirp, fs=SAMPLE_RATE, nperseg=256)

fig, ax = plt.subplots(figsize=(10, 4))
pcm = ax.pcolormesh(t_spec, f, 10 * np.log10(Sxx + 1e-12), shading="auto", cmap="viridis")
ax.set(xlabel="Time (s)", ylabel="Frequency (Hz)", title="Spectrogramme d'un chirp (5 → 200 Hz)")
plt.colorbar(pcm, ax=ax, label="dB")
plt.show()

## 5. Wavelets — analyse temps-frequence locale

In [ ]:
try:
    import pywt

    # Continuous Wavelet Transform sur s2
    scales = np.arange(1, 128)
    cwt_matr, freqs_cwt = pywt.cwt(s2[:1000], scales, "morl", sampling_period=1/SAMPLE_RATE)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.imshow(np.abs(cwt_matr), extent=[0, 1, freqs_cwt[-1], freqs_cwt[0]],
              cmap="jet", aspect="auto")
    ax.set(xlabel="Time (s)", ylabel="Frequency (Hz)", title="CWT de signal 2 (Morlet)")
    plt.show()

    # Discrete Wavelet Transform pour denoise
    coeffs = pywt.wavedec(s3, "db4", level=4)
    # Seuiller les coefficients (Universal threshold)
    threshold = np.std(coeffs[-1]) * np.sqrt(2 * np.log(len(s3)))
    coeffs_thresh = [coeffs[0]] + [pywt.threshold(c, threshold, mode="soft") for c in coeffs[1:]]
    s3_denoise = pywt.waverec(coeffs_thresh, "db4")[:len(s3)]

    fig, axes = plt.subplots(2, 1, figsize=(12, 5))
    axes[0].plot(t[:500], s3[:500]); axes[0].set_title("Bruite")
    axes[1].plot(t[:500], s3_denoise[:500]); axes[1].set_title("Apres wavelet denoising")
    plt.tight_layout(); plt.show()
except ImportError:
    print("pywavelets pas installe : uv add --group signal pywavelets")

## 6. PSD — Welch's method

In [ ]:
freqs_w, psd = sgn.welch(s3, fs=SAMPLE_RATE, nperseg=256)

fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(freqs_w, psd)
ax.set(xlabel="Frequency (Hz)", ylabel="PSD (V²/Hz)",
       title="Power Spectral Density (Welch's method)")
ax.grid(True)
plt.show()
print(f"Peaks visibles : 10, 50, 100 Hz")

## 7. Application — detection anomalie via energie freq

In [ ]:
# Synthese : 10 segments normaux (10 Hz + bruit) + 3 segments anormaux (haute frequence)
segments = []
labels = []
for i in range(13):
    if i in [3, 7, 11]:  # anormaux
        s = np.sin(2 * np.pi * 200 * t[:500]) + np.random.normal(0, 0.1, 500)
        labels.append("ANOMALY")
    else:
        s = np.sin(2 * np.pi * 10 * t[:500]) + np.random.normal(0, 0.1, 500)
        labels.append("NORMAL")
    segments.append(s)

# Feature : energie haute frequence (>100 Hz)
def high_freq_energy(signal, fs=SAMPLE_RATE, cutoff=100):
    fft_vals = np.abs(fft(signal))
    freqs = fftfreq(len(signal), 1/fs)
    mask = (freqs > cutoff) & (freqs < fs/2)
    return fft_vals[mask].mean()

energies = [high_freq_energy(s) for s in segments]
for i, (e, l) in enumerate(zip(energies, labels)):
    flag = "🚨" if e > np.mean(energies) + 2 * np.std(energies) else "  "
    print(f"  Segment {i:2d} : energy={e:7.2f}  ({l}) {flag}")

## 8. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| FFT sans windowing (rectangular) | Spectrum leakage — utiliser hamming/hann |
| Filter avec `lfilter` (causal) | Decalage de phase — `filtfilt` (zero-phase) |
| FFT sur sample size non-puissance de 2 | Plus lent (mais OK avec scipy.fft optimise) |
| Frequence > Nyquist (= fs/2) | Aliasing — toujours filtrer avant downsample |
| Spectrogramme avec `nperseg` trop petit | Mauvaise resolution frequentielle |
| Wavelet sans selection adaptee | Tester plusieurs (haar, db4, sym8, morl) |
| Pas verifier le bruit residuel apres denoise | Plot residus = signal - denoised |


## References

### Documentation
- scipy.signal : https://docs.scipy.org/doc/scipy/reference/signal.html
- PyWavelets : https://pywavelets.readthedocs.io/
- *Practical Time-Series Analysis* (Aileen Nielsen)
- *Think DSP* (Allen Downey) — gratuit : https://greenteapress.com/wp/think-dsp/

### Voir aussi
- [TS_Time_Series_Intro.ipynb](TS_Time_Series_Intro.ipynb)
- [TS_Maintenance_Predictive_GOOD.ipynb](TS_Maintenance_Predictive_GOOD.ipynb)
- [DL_PyTorch.ipynb](DL_PyTorch.ipynb)
